In [1]:
%matplotlib inline
import math
import time
import numpy as np
import torch
import torch.nn as nn
from d2l import torch as d2l

In [2]:
#vectorization speeds up the computation by using the parallelism of the hardware.
n = 10000
a = torch.ones(n)
b = torch.ones(n)

In [ ]:
c = torch.zeros(n)
t = time.time()
for i in range(n):
    c[i] = a[i] + b[i] #Non-vectorized addition
print('%.5f sec' % (time.time() - t))

0.28440 sec


In [ ]:
t = time.time()
d = a + b #Vectorized addition
print('%.5f sec' % (time.time() - t))

0.00000 sec


In [4]:
#Utility Functions

def add_to_class(Class): #@save
    """Register function as methods in created class."""
    def wrapper(obj):
        setattr(Class, obj.__name__, obj)
    return wrapper

In [2]:
#Utility Classes

class HyperParameters: #@save
    """A class to manage hyperparameters."""
    def save_hyperparameters(self, ignore=[]):
        raise NotImplementedError
    
class ProgressBoard(d2l.HyperParameters): #@save
    """The progress board that plots data points in animation."""
    def __init__(self, xlabel=None, ylabel=None, xlim=None, ylim=None, 
                 xscale='linear', yscale='linear', ls=['-', '--', '-.', ':'], colors=['C0', 'C1', 'C2', 'C3'], 
                 fig=None, axes=None, figsize=(3.5, 2.5), display=True):
        self.save_hyperparameters()

    def draw(self, x, y, label, every_n=1):
        raise NotImplementedError

In [3]:
#The Module class is the base class for all neural network modules. Your models should also subclass this class.
class Module(nn.Module, d2l.HyperParameters): #@save
    """The base class for all neural network models."""
    def __init__(self, plot_train_per_epoch=2, plot_valid_per_epoch=1):
        super().__init__()
        self.save_hyperparameters()
        self.board = ProgressBoard()

    def loss(self, y_hat, y):
        raise NotImplementedError
    
    def forward(self, X):
        assert hasattr(self, 'net'), 'Neural network is not defined.'
        return self.net(X)
    
    def plot(self, key, value, train):
        """Plot a point in animation."""
        assert hasattr(self, 'trainer'), 'Trainer is not initiated.'
        self.board.xlabel = 'epoch'
        if train:
            x = self.trainer.train_batch_idx / \
                self.trainer.num_train_batches
            n = self.trainer.num_train_batches / \
                self.plot_train_per_epoch
        else:
            x = self.trainer.epoch + 1
            n = self.trainer.num_val_batches / \
                self.plot_valid_per_epoch
        self.board.draw(x, value.to(d2l.cpu()).deatch().numpy(), 
                        ('train_' if train else 'val_') + key, every_n=int(n))
        
    def training_step(self, batch):
        """The training step defined by loss function."""
        l = self.loss(self(*batch[:-1]), batch[-1])
        self.plot('loss', l, train=True)
        return l
    
    def validation_step(self, batch):
        """The validation step defined by loss function."""
        l = self.loss(self(*batch[:-1]), batch[-1])
        self.plot('loss', l, train=False)
        return l
    
    def configure_optimizers(self):
        raise NotImplementedError

In [4]:
class DataModule(d2l.HyperParameters): #@save
    """The base class for data modules."""
    def __init__(self, root='../data', num_workers=4):
        self.save_hyperparameters()

    def get_dataloader(self, train):
        raise NotImplementedError
    
    def train_dataloader(self):
        return self.get_dataloader(train=True)
    
    def val_dataloader(self):
        return self.get_dataloader(train=False)

In [5]:
class Trainer(d2l.HyperParameters): #@save
    """The trainer that trains and evaluates a model."""
    def __init__(self, max_epochs, num_gpus=0, gradient_clip_val=0):
        self.save_hyperparameters()
        assert num_gpus == 0, 'No GPU support yet.'

    def prepare_data(self, data):
        """Prepare the dataloader."""
        self.train_dataloader = data.train_dataloader()
        self.val_dataloader = data.val_dataloader()
        self.num_train_batches = len(self.train_dataloader)
        self.num_val_batches = len(self.val_dataloader if self.val_dataloader is not None else [])

    def prepare_model(self, model):
        """Prepare the model and its optimizer."""
        model.trainer = self
        model.board.xlim = [0, self.max_epochs]
        self.model = model

    def fit(self, data, model):
        self.prepare_data(data)
        self.prepare_model(model)
        self.optim = model.configure_optimizers()
        self.epoch = 0
        self.train_batch_idx = 0
        self.val_batch_idx = 0
        for self.epoch in range(self.max_epochs):
            self.fit_epoch()

    def fit_epoch(self):
        raise NotImplementedError